# 06.03 — Lag Features

Create historical target lags within each FSA.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'feature_engineering.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/feature_engineering.yaml')

In [3]:
# Import module to manage the Feature Engineering process
from src.ontario_peak_risk.feature_engineering.common import (
    load_feature_config,
    load_clean_dataset,
    ensure_directories,
)

In [4]:
# Config Feature Engineering process and Load the dataset
CONFIG, _ = load_feature_config(CONFIG_PATH)
REPORTS_DIR, DOCS_DIR = ensure_directories(CONFIG, PROJECT_ROOT)
clean_dataset = load_clean_dataset(CONFIG, PROJECT_ROOT)
clean_dataset.shape

(262944, 66)

In [5]:
# Import the functions to run baseline Feature Engineering process and add temporal features
from src.ontario_peak_risk.feature_engineering.base_features import build_base_dataset
from src.ontario_peak_risk.feature_engineering.temporal_features import add_temporal_features
from src.ontario_peak_risk.feature_engineering.lag_features import add_target_lags

In [6]:
# Run the baseline Feature Engineering process
base_dataset, _ = build_base_dataset(clean_dataset, CONFIG)
temporal_dataset, _ = add_temporal_features(base_dataset, CONFIG)
lag_dataset, lag_dictionary = add_target_lags(temporal_dataset, CONFIG)
lag_dictionary

,feature_name,feature_family,source_columns,description,forecasting_candidate,peak_risk_candidate,computation_scope,leakage_risk
0,total_consumption_kwh_lag_1h,target_lag,total_consumption_kwh,Electricity consumption observed 1 hour(s) bef...,True,True,historical_only,low_if_computed_after_temporal_ordering
1,total_consumption_kwh_lag_2h,target_lag,total_consumption_kwh,Electricity consumption observed 2 hour(s) bef...,True,True,historical_only,low_if_computed_after_temporal_ordering
2,total_consumption_kwh_lag_3h,target_lag,total_consumption_kwh,Electricity consumption observed 3 hour(s) bef...,True,True,historical_only,low_if_computed_after_temporal_ordering
3,total_consumption_kwh_lag_24h,target_lag,total_consumption_kwh,Electricity consumption observed 24 hour(s) be...,True,True,historical_only,low_if_computed_after_temporal_ordering
4,total_consumption_kwh_lag_48h,target_lag,total_consumption_kwh,Electricity consumption observed 48 hour(s) be...,True,True,historical_only,low_if_computed_after_temporal_ordering
5,total_consumption_kwh_lag_168h,target_lag,total_consumption_kwh,Electricity consumption observed 168 hour(s) b...,True,True,historical_only,low_if_computed_after_temporal_ordering


In [7]:
# Display the lag dataset with the new features
lag_columns = [c for c in lag_dataset.columns if '_lag_' in c]
lag_dataset[['fsa','timestamp','total_consumption_kwh',*lag_columns]].head(10)

,fsa,timestamp,total_consumption_kwh,total_consumption_kwh_lag_1h,total_consumption_kwh_lag_2h,total_consumption_kwh_lag_3h,total_consumption_kwh_lag_24h,total_consumption_kwh_lag_48h,total_consumption_kwh_lag_168h
0,L4T,2021-01-01 00:00:00,10276.6,NaN,NaN,NaN,NaN,NaN,NaN
1,L4T,2021-01-01 01:00:00,9585.1,10276.6,NaN,NaN,NaN,NaN,NaN
2,L4T,2021-01-01 02:00:00,9015.5,9585.1,10276.6,NaN,NaN,NaN,NaN
3,L4T,2021-01-01 03:00:00,8593.1,9015.5,9585.1,10276.6,NaN,NaN,NaN
4,L4T,2021-01-01 04:00:00,8353.5,8593.1,9015.5,9585.1,NaN,NaN,NaN
5,L4T,2021-01-01 05:00:00,8334.4,8353.5,8593.1,9015.5,NaN,NaN,NaN
6,L4T,2021-01-01 06:00:00,8499.5,8334.4,8353.5,8593.1,NaN,NaN,NaN
7,L4T,2021-01-01 07:00:00,8784.2,8499.5,8334.4,8353.5,NaN,NaN,NaN
8,L4T,2021-01-01 08:00:00,9313.2,8784.2,8499.5,8334.4,NaN,NaN,NaN
9,L4T,2021-01-01 09:00:00,10062.2,9313.2,8784.2,8499.5,NaN,NaN,NaN
